**Quest 13: Never Gonna Let You Down**

![Header image](./imgs/image_quest_13.svg)
[<img src="./imgs/instagram.svg" alt="My SVG" width="15" height="15"><small>_Monika Lipińska_</small>](https://www.instagram.com/monli_art/)

---

## Part I — Story

A new day begins and your festivities are interrupted by a messenger arriving at the camp. He brings news from the king that the dear **Princess Adventa** of the Royal Code Family and her feline companion have been captured and are in dire need of rescuing, without delay.

---

## The Challenge

The princess is being held in the **Chamber of Eternal Descent**, an area filled with walls and floating platforms that shift between levels.

---

## Rules of the Chamber

- **Movement:** You may move between platforms only if they are on the **same level**.
- **Levels:** Platforms exist at **10 levels**: 0 through 9.
- **Portals:** Levels wrap around — moving up from 9 goes to 0; moving down from 0 goes to 9.
- **Walls:** Marked with `#` and cannot be crossed.
- **One‑Way Trip:** Each platform disintegrates after you leave it; **no revisiting** squares.
- **Timing:**
  - Changing the level of your current platform costs **1 second per level** (shortest wrap‑around distance).
  - Moving horizontally to another platform costs **1 second**.
- **Goal:** Start at **S** (level 0) and reach **E** (level 0) in the **minimum possible time**.

---

## Example

Given the following maze:

```
#######
#6769##
S50505E
#97434#
#######
```

### Optimal Path Analysis

- Some paths take **36 seconds** or **30 seconds**.
- The **optimal** path takes **28 seconds**.

---

## Your Task

Determine the **minimum number of seconds** required to complete **your** maze.

**Note:** Level‑change time uses the shortest wrap‑around distance.  
Example:

- From level 2 → 5 costs **3 seconds**
- From level 0 → 9 costs **1 second** (wrap‑around)

---


In [16]:
from collections.abc import Iterator
from heapq import heappop, heappush
from itertools import product

from more_itertools import one
from test_utilities import test

tests = [
    {
        "name": "Example Part I",
        "notes": """
            #######
            #6769##
            S50505E
            #97434#
            #######
        """,
        "expected": 28,
    },
]


class Grid:
    def __init__(self, notes: str) -> None:
        self.grid = [
            [int(c) if c.isdigit() else c for c in row.lstrip()]
            for row in notes.strip().splitlines()
        ]

        self.rows, self.cols = len(self.grid), len(self.grid[0])

        self.S = self.get("S")

        for r, c in self.S:
            self.grid[r][c] = 0

        self.E = one(self.get("E"))

        self.grid[self.E[0]][self.E[1]] = 0

    def get(self, value: str) -> list[tuple[int, int]]:
        return [
            (r, c)
            for r, c in product(range(self.rows), range(self.cols))
            if self.grid[r][c] == value
        ]

    def minimum_seconds(self) -> int:
        heap = [(0, r, c) for r, c in self.S]
        seen = set()

        while heap:
            t, r, c = heappop(heap)

            if (r, c) == self.E:
                return t

            if (r, c) in seen:
                continue

            seen.add((r, c))

            level = self.grid[r][c]  # level can only be int

            for r_n, c_n in self.neighbors(r, c):
                level_n = self.grid[r_n][c_n]
                if isinstance(level_n, int):

                    dt = self.dist_mod10(level, level_n) + 1  # type: ignore
                    heappush(heap, (t + dt, r_n, c_n))

        raise ValueError("No path")

    @staticmethod
    def dist_mod10(a: int, b: int) -> int:
        diff = abs(a - b)
        return min(diff, 10 - diff)

    def neighbors(self, row: int, col: int) -> Iterator[tuple[int, int]]:

        for dr, dc in ((-1, 0), (0, 1), (1, 0), (0, -1)):
            if 0 <= row + dr < self.rows and 0 <= col + dc < self.cols:
                yield row + dr, col + dc

    def __str__(self) -> str:
        return "\n".join("".join(f"{c}" for c in r) for r in self.grid)


@test(tests=tests[:])
def part_I(notes: str) -> int:
    grid = Grid(notes)
    return grid.minimum_seconds()


Test Example Part I passed, for part_I.
Success


In [17]:
with open("../inputs/everybody_codes_e2024_q13_p1.txt") as f:
    notes1 = f.read()

print(f"Part I: {part_I(notes1)}")

Part I: 169


## Part II

Success! You have secured the princess and her pet, and now it is just the simple matter of exiting the chamber. That should be easy, right? You can't return the way you came, so you enter the next, much larger, maze.
Fortunately, the rules remain unchanged, so you can swiftly devise another action plan. The staring point is still marked with S and the exit on the further end is marked with an E .


In [18]:
with open("../inputs/everybody_codes_e2024_q13_p2.txt") as f:
    notes2 = f.read()

print(f"PartII: {part_I(notes2)}")

PartII: 640


### Part III

You swiftly moved through the maze only to discover that there is yet another before you. However, it looks somewhat different. At its center is a special portal on level 0, marked as E , that can be used as an exit to the outside. It is surrounded by many walls and platforms at various levels.
The entire area is enclosed by platforms at level 0, marked as S . You can start the mission from any S point on this floor, but which starting position will allow you to do it as fast as possible?
**Example** based on the following notes:

```
SSSSSSSSSSS
S674345621S
S###6#4#18S
S53#6#4532S
S5450E0485S
S##7154532S
S2##314#18S
S971595#34S
SSSSSSSSSSS
```

With the optimal path, this maze can be completed in 14 sec.

```
SSSSSSSSSSS
S674345621S
S###6#4#18S
S53#6#4532S
S5450E0485S
S##7154532S
S2##314#18S
S971595#34S
SSSSSSSSSSS
```

> What is the minimum number of seconds needed to complete your maze?


In [19]:
from test_utilities import test

tests = [
    {
        "name": "Example Part I",
        "notes": """
            SSSSSSSSSSS
            S674345621S
            S###6#4#18S
            S53#6#4532S
            S5450E0485S
            S##7154532S
            S2##314#18S
            S971595#34S
            SSSSSSSSSSS
        """,
        "expected": 14,
    },
]


@test(tests=tests[:])
def part_I(notes: str) -> int:
    grid = Grid(notes)
    return grid.minimum_seconds()


Test Example Part I passed, for part_I.
Success


In [20]:
with open("../inputs/everybody_codes_e2024_q13_p3.txt") as f:
    notes3 = f.read()

print(f"Part III: {part_I(notes3)}")

Part III: 583


![happy](./imgs/happy_quack.svg)
